# Día 10 · Unidad 12 — Cálculo de rentas en Python

**Objetivos de aprendizaje**
- Implementar el valor actual de una renta constante, temporal y creciente como funciones de Python con NumPy.
- Validar cada fórmula cerrada comparándola con una suma explícita de flujos descontados.

**Duración estimada:** 45-60 minutos.

**Requisitos previos:** ninguno específico de Python — el contenido actuarial-financiero de esta sección (rentas, valor actual, tipo de descuento) se asume ya dominado. Aquí el foco está en el código, no en la teoría.

---

Esta unidad (12) no tiene material previo en el curso: se ha escrito íntegramente para esta edición. Dado el nivel del público (colegiados con formación actuarial y financiera sólida), la teoría se trata de forma breve y el peso está en la implementación en Python y en la validación numérica de cada resultado — el mismo criterio de "valida siempre, no des nada por hecho" que pide `docs/uso_ia_en_el_curso.md`, aplicado aquí a fórmulas cerradas en vez de a código generado por IA.

## 1. Renta constante (temporal, pospagable)

Valor actual de $n$ flujos de importe constante $C$, con tipo de interés efectivo $i$ por periodo, el primero pagadero al final del primer periodo:

$$PV = C \cdot \frac{1 - (1+i)^{-n}}{i}$$

Es la fórmula que ya usáis para valorar una pensión temporal, una amortización de préstamo (vista desde el lado del prestamista) o el valor actual de un flujo de primas futuras fijas.

In [1]:
import numpy as np


def pv_renta_constante(C: float, i: float, n: int) -> float:
    """Valor actual de una renta constante, temporal, pospagable.

    Args:
        C: importe constante de cada flujo.
        i: tipo de interes efectivo por periodo (p. ej. 0.03 para 3%).
        n: numero de periodos (flujos).

    Returns:
        float: valor actual de los n flujos.
    """
    return C * (1 - (1 + i) ** (-n)) / i

**Validación**: comparamos la fórmula cerrada con la suma explícita de cada flujo descontado, flujo a flujo — el caso de control más directo posible.

In [2]:
C, i, n = 1000.0, 0.03, 15

pv_formula = pv_renta_constante(C, i, n)

periodos = np.arange(1, n + 1)
pv_suma_directa = np.sum(C / (1 + i) ** periodos)

print(f"PV (formula cerrada):   {pv_formula:.6f}")
print(f"PV (suma de flujos):    {pv_suma_directa:.6f}")
assert np.isclose(pv_formula, pv_suma_directa), "La formula cerrada no coincide con la suma directa de flujos"
print("Coinciden.")

PV (formula cerrada):   11937.935087
PV (suma de flujos):    11937.935087
Coinciden.


## 2. Perpetuidad constante

Caso límite cuando $n \to \infty$: el término $(1+i)^{-n} \to 0$, así que:

$$PV_\infty = \frac{C}{i}$$

In [3]:
def pv_perpetuidad(C: float, i: float) -> float:
    """Valor actual de una perpetuidad constante, pospagable.

    Args:
        C: importe constante de cada flujo, indefinidamente.
        i: tipo de interes efectivo por periodo.

    Returns:
        float: valor actual de la perpetuidad.
    """
    return C / i


# Validacion: con n muy grande, la renta temporal debe converger a la perpetuidad
pv_temporal_grande = pv_renta_constante(C, i, n=100_000)
pv_perp = pv_perpetuidad(C, i)
print(f"PV renta temporal (n=100.000): {pv_temporal_grande:.4f}")
print(f"PV perpetuidad:                {pv_perp:.4f}")
assert np.isclose(pv_temporal_grande, pv_perp, atol=1e-6)
print("Convergen, como se esperaba.")

PV renta temporal (n=100.000): 33333.3333
PV perpetuidad:                33333.3333
Convergen, como se esperaba.


## 3. Renta creciente (geométrica) y perpetuidad creciente

Si cada flujo crece a una tasa constante $g$ respecto al anterior ($C, C(1+g), C(1+g)^2, \dots$), y $i \neq g$:

$$PV = C \cdot \frac{1 - \left(\frac{1+g}{1+i}\right)^n}{i - g}$$

Si $i = g$, la fórmula anterior tiene una indeterminación $0/0$ — el caso límite se resuelve por L'Hôpital y da $PV = C \cdot n / (1+i)$, que hay que tratar aparte en el código.

In [4]:
def pv_renta_creciente(C: float, i: float, g: float, n: int) -> float:
    """Valor actual de una renta geometricamente creciente, temporal, pospagable.

    Args:
        C: importe del primer flujo.
        i: tipo de interes efectivo por periodo.
        g: tasa de crecimiento constante entre flujos sucesivos.
        n: numero de periodos (flujos).

    Returns:
        float: valor actual de los n flujos crecientes.
    """
    if np.isclose(i, g):
        return C * n / (1 + i)
    return C * (1 - ((1 + g) / (1 + i)) ** n) / (i - g)


def pv_perpetuidad_creciente(C: float, i: float, g: float) -> float:
    """Valor actual de una perpetuidad geometricamente creciente (modelo de Gordon).

    Args:
        C: importe del primer flujo.
        i: tipo de interes efectivo por periodo. Debe ser mayor que g para que converja.
        g: tasa de crecimiento constante entre flujos sucesivos.

    Returns:
        float: valor actual de la perpetuidad creciente.
    """
    if i <= g:
        raise ValueError("La perpetuidad creciente solo converge si i > g")
    return C / (i - g)

**Validación 1**: de nuevo contra la suma explícita de flujos, esta vez con crecimiento geométrico.

In [5]:
C, i, g, n = 1000.0, 0.05, 0.02, 20

pv_formula = pv_renta_creciente(C, i, g, n)

periodos = np.arange(0, n)  # el primer flujo (periodo 1) crece C*(1+g)^0
flujos = C * (1 + g) ** periodos
pv_suma_directa = np.sum(flujos / (1 + i) ** (periodos + 1))

print(f"PV (formula cerrada): {pv_formula:.6f}")
print(f"PV (suma de flujos):  {pv_suma_directa:.6f}")
assert np.isclose(pv_formula, pv_suma_directa)
print("Coinciden.")

PV (formula cerrada): 14665.401812
PV (suma de flujos):  14665.401812
Coinciden.


**Validación 2**: el caso límite $i = g$ (indeterminación resuelta a mano en el código) frente a la suma directa de flujos con $i$ y $g$ iguales.

In [6]:
C, i_igual_g, n = 1000.0, 0.04, 10

pv_formula_limite = pv_renta_creciente(C, i_igual_g, i_igual_g, n)

periodos = np.arange(0, n)
flujos = C * (1 + i_igual_g) ** periodos
pv_suma_directa = np.sum(flujos / (1 + i_igual_g) ** (periodos + 1))

print(f"PV (caso limite i=g, formula):  {pv_formula_limite:.6f}")
print(f"PV (caso limite i=g, suma):     {pv_suma_directa:.6f}")
assert np.isclose(pv_formula_limite, pv_suma_directa)
print("Coinciden.")

PV (caso limite i=g, formula):  9615.384615
PV (caso limite i=g, suma):     9615.384615
Coinciden.


**Validación 3**: la renta creciente con `g=0` debe coincidir exactamente con la renta constante de la sección 1 — es un caso particular.

In [7]:
C, i, n = 1000.0, 0.03, 15

pv_creciente_g0 = pv_renta_creciente(C, i, g=0.0, n=n)
pv_constante = pv_renta_constante(C, i, n)

print(f"PV renta creciente con g=0: {pv_creciente_g0:.6f}")
print(f"PV renta constante:         {pv_constante:.6f}")
assert np.isclose(pv_creciente_g0, pv_constante)
print("Coinciden: la renta constante es el caso particular g=0 de la renta creciente.")

PV renta creciente con g=0: 11937.935087
PV renta constante:         11937.935087
Coinciden: la renta constante es el caso particular g=0 de la renta creciente.


## 4. Aplicación: valor actual de un flujo de primas futuras

Ejemplo de uso conjunto: una cartera de pólizas de vida que se espera genere una prima anual de 250.000€, con una tasa de caída de cartera (lapse) que hace que el flujo esperado **decrezca** un 3% anual (crecimiento negativo, $g = -0.03$), a lo largo de un horizonte de valoración de 10 años, descontado a un tipo del 4%.

In [8]:
prima_anual_esperada = 250_000.0
tipo_descuento = 0.04
tasa_caida_cartera = -0.03  # g negativo: el flujo esperado decrece
horizonte_anios = 10

valor_actual_primas = pv_renta_creciente(
    C=prima_anual_esperada, i=tipo_descuento, g=tasa_caida_cartera, n=horizonte_anios
)
print(f"Valor actual del flujo de primas esperado: {valor_actual_primas:,.2f} EUR")

# Para ti: compara este resultado con el de tratar el flujo como constante
# (ignorando la caida de cartera) y valora cuanto se estaria sobrestimando.
valor_actual_sin_caida = pv_renta_constante(prima_anual_esperada, tipo_descuento, horizonte_anios)
diferencia_pct = (valor_actual_sin_caida / valor_actual_primas - 1) * 100
print(f"Valor actual ignorando la caida de cartera: {valor_actual_sin_caida:,.2f} EUR ({diferencia_pct:.1f}% mas alto)")

Valor actual del flujo de primas esperado: 1,792,223.87 EUR
Valor actual ignorando la caida de cartera: 2,027,723.94 EUR (13.1% mas alto)


**Para ti:** implementa `pv_renta_prepagable(C, i, n)` (flujos al **inicio** de cada periodo, no al final). Pista: es exactamente `pv_renta_constante(C, i, n) * (1 + i)` — demuéstralo con el caso de control de la sección 1.

**Para ti:** con `pv_perpetuidad_creciente`, comprueba qué pasa si llamas a la función con `i <= g` (por ejemplo `i=0.03, g=0.05`). ¿Por qué tiene sentido financiero que la función lance un error en vez de devolver un número, por muy "válido" que parezca aritméticamente el cálculo si se le quita el `if`?

## Resumen

- Toda fórmula cerrada de renta se puede (y se debe) validar contra la suma explícita de flujos descontados con NumPy — es el equivalente de "cuadrar a mano" antes de confiar en una fórmula.
- `pv_renta_constante` es el caso base; `pv_perpetuidad` es su límite cuando $n \to \infty$; `pv_renta_creciente` generaliza ambas ($g=0$ recupera la renta constante).
- El caso $i = g$ en la renta creciente es una indeterminación matemática que hay que resolver explícitamente en el código (no dejar que NumPy divida por cero silenciosamente).
- Implementar estas fórmulas en funciones puras (sin estado, con `docstring` y validación por caso de control) es exactamente el tipo de tarea donde un asistente de IA puede acelerar la escritura del código una vez decidida la fórmula — pero la fórmula, el supuesto de tipo de interés y la interpretación del resultado siguen siendo criterio actuarial vuestro, no de la IA.

**Siguiente:** `05_teoria_opciones_montecarlo.ipynb` — simulación Monte Carlo de una opción europea, contrastada con Black-Scholes.